# 01 EDA: まずデータを確認する(S6E10 Predicting Airline Satisfaction)

目的: 特徴量を考える前に、データの形・target・欠損・train/test の違いを押さえる。

- 1セル = 1つの問い。各問いの下の「見えたこと」は自分で書く(事実 / 比較 / 理由の仮説 / 次の行動)
- 図は `eda_figs/` に保存される

## 準備: 読み込み

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA = Path("/kaggle/input/playground-series-s6e10")
ID, TARGET = "id", "satisfaction"
TRAIN_C, TEST_C = "#2a78d6", "#eb6834"  # train=青, test=橙(全図で共通)。図中の文字は英語(Kaggleに日本語フォントがないため)
FIG = Path("eda_figs")
FIG.mkdir(exist_ok=True)

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print([p.name for p in DATA.iterdir()])  # 配布ファイル一覧(original データの有無も確認)

## Q1. データの大きさと列は? train と test で列は揃っているか?
なぜ見るか: 行数で使える手法・CV の重さが決まる。列の差が target 以外にあれば前処理で揃える必要がある。

予想: test は train に比べて target 列がないだけ。

In [ ]:
print("train:", train.shape, "/ test:", test.shape)
print("train にだけある列:", sorted(set(train.columns) - set(test.columns)))
print("test にだけある列:", sorted(set(test.columns) - set(train.columns)))
train.head()

**見えたこと**(ここに書く)
- 事実:
- 比較:
- 理由の仮説:
- 次の行動:

## Q2. 各列の型とユニーク数は?(どれがカテゴリ列か)
なぜ見るか: カテゴリ列の扱い(category 型 / エンコード)と、数値に見えて実は段階評価の列を見分けるため。

予想: 文字列のカテゴリ列が数本、0〜5 の満足度評価の整数列が多数、遅延や距離などの連続値が少し。

In [ ]:
cols = [c for c in test.columns if c != ID]
info = pd.DataFrame({
    "dtype": train[cols].dtypes.astype(str),
    "nunique_train": train[cols].nunique(),
    "nunique_test": test[cols].nunique(),
    "例": [train[c].dropna().unique()[:6].tolist() for c in cols],
})
info

以降は「文字列の列 + ユニーク数が10以下の数値列」をカテゴリ扱い、それ以外を連続値として見る。

In [ ]:
cat_cols = [c for c in cols if train[c].dtype == "object" or train[c].nunique() <= 10]
num_cols = [c for c in cols if c not in cat_cols]
print("カテゴリ扱い:", cat_cols)
print("連続値:", num_cols)

**見えたこと**(ここに書く)
- 事実:
- 比較:
- 理由の仮説:
- 次の行動:

## Q3. target の値は何で、割合はどうか?
なぜ見るか: 陽性ラベルの文字列を確定させて config の `positive` を合わせるため(今は `satisfied` と仮置き)。偏りが強いかも確認する。

予想: `satisfied` と `neutral or dissatisfied` の2値で、元データと同じく dissatisfied がやや多い(約 57:43)。

In [ ]:
vc = train[TARGET].value_counts(dropna=False)
print(pd.DataFrame({"件数": vc, "割合": (vc / len(train)).round(4)}))

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(vc.index.astype(str), vc.values / len(train), color=TRAIN_C, height=0.5)
for i, v in enumerate(vc.values / len(train)):
    ax.text(v + 0.01, i, f"{v:.1%}", va="center")
ax.set_xlim(0, 1)
ax.set_xlabel("share of train")
ax.set_title(f"{TARGET} distribution (train, n={len(train):,})")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG / "q3_target.png", dpi=120)
plt.show()

**見えたこと**(ここに書く)
- 事実:
- 比較:
- 理由の仮説:
- 次の行動:

## Q4. 欠損はどの列にどれくらいあるか? train と test で同じか?
なぜ見るか: 欠損の扱い(そのまま LightGBM に任せる / 補完 / 欠損フラグ)を決めるため。train と test で率が違えば要注意。

予想: 元データでは Arrival Delay だけ 0.3% 程度欠損。合成データなので他の列にも少し入っているかもしれない。

In [ ]:
miss = pd.DataFrame({
    "train": train[cols].isna().mean(),
    "test": test[cols].isna().mean(),
})
print(miss[(miss > 0).any(axis=1)].sort_values("train", ascending=False).round(4))

m = miss.sort_values("train")
y = np.arange(len(m))
fig, ax = plt.subplots(figsize=(7, max(3, 0.3 * len(m))))
ax.barh(y + 0.2, m["train"] * 100, height=0.38, color=TRAIN_C, label="train")
ax.barh(y - 0.2, m["test"] * 100, height=0.38, color=TEST_C, label="test")
ax.set_yticks(y, m.index)
ax.set_xlabel("missing rate (%)")
ax.set_title("Missing rate by column")
ax.legend(loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG / "q4_missing.png", dpi=120)
plt.show()

**見えたこと**(ここに書く)
- 事実:
- 比較:
- 理由の仮説:
- 次の行動:

## Q5. 連続値の列の分布は? train と test でズレていないか?
なぜ見るか: 外れ値・裾の長さ(log 変換が要るか)と、train/test の分布のズレ(CV と LB が乖離する原因)を確認するため。

予想: 遅延時間は 0 に集中して右に長い裾。train と test はほぼ重なる。

In [ ]:
n = len(num_cols)
ncol = 3
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.5 * ncol, 3 * nrow), squeeze=False)
for ax, c in zip(axes.flat, num_cols):
    lo, hi = np.nanpercentile(pd.concat([train[c], test[c]]), [0, 99.5])
    bins = np.linspace(lo, hi, 50)
    ax.hist(train[c].dropna(), bins=bins, density=True, histtype="step", lw=2, color=TRAIN_C, label="train")
    ax.hist(test[c].dropna(), bins=bins, density=True, histtype="step", lw=2, color=TEST_C, label="test")
    ax.set_title(f"{c}\n(max={train[c].max():,.0f}, top 0.5% clipped)", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes.flat[n:]:
    ax.axis("off")
axes.flat[0].legend()
plt.tight_layout()
plt.savefig(FIG / "q5_numeric_dist.png", dpi=120)
plt.show()

train[num_cols].describe().T

**見えたこと**(ここに書く)
- 事実:
- 比較:
- 理由の仮説:
- 次の行動:

## Q6. カテゴリ列の分布は? train と test でズレていないか?
なぜ見るか: 少数カテゴリ・test にしかない値の有無と、train/test の比率の差を確認するため。

予想: 評価(0〜5)は 3〜4 が多い。train と test の比率はほぼ同じ。

In [ ]:
n = len(cat_cols)
ncol = 3
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.5 * ncol, 3 * nrow), squeeze=False)
for ax, c in zip(axes.flat, cat_cols):
    p = pd.DataFrame({
        "train": train[c].astype(str).value_counts(normalize=True),
        "test": test[c].astype(str).value_counts(normalize=True),
    }).fillna(0).sort_index()
    x = np.arange(len(p))
    ax.bar(x - 0.2, p["train"], width=0.38, color=TRAIN_C, label="train")
    ax.bar(x + 0.2, p["test"], width=0.38, color=TEST_C, label="test")
    ax.set_xticks(x, p.index, rotation=30 if p.index.str.len().max() > 4 else 0, ha="right", fontsize=8)
    ax.set_title(c, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes.flat[n:]:
    ax.axis("off")
axes.flat[0].legend()
plt.tight_layout()
plt.savefig(FIG / "q6_category_dist.png", dpi=120)
plt.show()

only_test = {c: sorted(set(test[c].dropna().astype(str)) - set(train[c].dropna().astype(str))) for c in cat_cols}
print("test にだけある値:", {c: v for c, v in only_test.items() if v} or "なし")

**見えたこと**(ここに書く)
- 事実:
- 比較:
- 理由の仮説:
- 次の行動:

## Q7. 重複行はあるか?(id を除いて)
なぜ見るか: 合成データでは同じ特徴量の行が多いことがある。train 内で同じ特徴量なのに target が違う行の割合は、AUC の上限の目安になる。

予想: 連続値が多いので重複はほぼない。

In [ ]:
feat = cols
dup_train = train.duplicated(subset=feat).sum()
dup_test = test.duplicated(subset=feat).sum()
key_train = set(map(tuple, train[feat].astype(str).values))
in_train = test[feat].astype(str).apply(tuple, axis=1).isin(key_train).sum()
conflict = (train.groupby(feat, dropna=False)[TARGET].nunique() > 1).sum()
print(f"train 内の重複行: {dup_train:,} ({dup_train / len(train):.2%})")
print(f"test 内の重複行: {dup_test:,} ({dup_test / len(test):.2%})")
print(f"test のうち train と同じ特徴量の行: {in_train:,} ({in_train / len(test):.2%})")
print(f"同じ特徴量で target が食い違うグループ数: {conflict:,}")

**見えたこと**(ここに書く)
- 事実:
- 比較:
- 理由の仮説:
- 次の行動:

## まとめ(EDA後に書く)
- `positive` の値: 
- カテゴリ列 / 欠損 / 外れ値で決めたこと:
- 次に試す特徴量: